# Chest X-ray Grad-CAM 시각화

DenseNet121 기반 U-multiclass 모델의 예측 결과를 Grad-CAM으로 시각화하여, 각 소견의 예측에 기여한 이미지 영역을 확인한다.

### 탐지 대상
- 심비대 (Cardiomegaly)
- 폐부종 (Edema)
- 흉수 (Pleural Effusion)

### 라벨 규칙
- 0: 음성 (Negative)
- 1: 양성 (Positive)
- 2: 불확실 (Uncertain)

### 시각화 구성
하나의 흉부 X-ray 이미지에 대해 총 4개의 이미지를 출력한다.

1. 원본 X-ray
2. 심비대 Grad-CAM Overlay
3. 폐부종 Grad-CAM Overlay
4. 흉수 Grad-CAM Overlay

## 2. 라이브러리 및 실행 환경

Grad-CAM 구현에 필요한 라이브러리를 불러오고 CPU/GPU 실행 환경을 설정한다.

- PyTorch: 모델 연산 및 기울기 계산
- torchvision: DenseNet121 모델 구성
- NumPy: 이미지 및 히트맵 배열 처리
- Matplotlib: Grad-CAM 결과 시각화

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn
from torchvision.models import densenet121, DenseNet121_Weights

import numpy as np
import matplotlib.pyplot as plt

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

FINDING_NAMES = ["Cardiomegaly", "Edema", "Pleural Effusion"]
CLASS_NAMES = ["Negative", "Positive", "Uncertain"]

print("사용 장치:", device)

## 3. DenseNet121 모델 정의

기존 모델 학습 코드와 동일한 DenseNet121 기반 U-multiclass 모델 구조를 사용한다.

- Backbone: DenseNet121
- 출력 형태: (B, 3, 3)
- 소견 수: 3개
- 클래스 수: 소견별 3개

In [ ]:
class CXRModel(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()

        weights = DenseNet121_Weights.DEFAULT if pretrained else None

        self.backbone = densenet121(weights=weights)
        self.backbone.classifier = nn.Linear(
            self.backbone.classifier.in_features, 9
        )

    def forward(self, images):
        features = self.backbone.features(images)

        # 특징 지도를 제자리에서 변경하지 않도록 설정
        features = F.relu(features, inplace=False)

        pooled = F.adaptive_avg_pool2d(features, (1, 1))
        flattened = torch.flatten(pooled, 1)
        logits = self.backbone.classifier(flattened)

        return logits.reshape(-1, 3, 3)

## 4. Grad-CAM 구현

Grad-CAM(Gradient-weighted Class Activation Mapping)을 이용하여 특정 소견과 클래스의 예측 점수에 기여한 영역을 시각화한다.

### 구현 과정

1. DenseNet121의 마지막 특징 추출 계층에서 Feature Map 추출
2. 선택한 소견 및 클래스의 출력 점수에 대한 Gradient 계산
3. Gradient의 공간 평균으로 채널별 가중치 계산
4. Feature Map과 가중치를 결합하여 CAM 생성
5. ReLU 적용 및 입력 이미지 크기로 확대
6. 히트맵을 0~1 범위로 정규화

### Grad-CAM 대상

- 소견 인덱스: 0=심비대, 1=폐부종, 2=흉수
- 기본 클래스 인덱스: 1=양성

Grad-CAM은 모델 예측에 기여한 영역을 보여주며, 실제 병변 위치를 정확히 나타내는 것은 아니다.

In [ ]:
class GradCAM:
    def __init__(self, model):
        self.model = model
        self.target_layer = model.backbone.features.norm5

        self.activations = None
        self.gradients = None

        self.forward_handle = self.target_layer.register_forward_hook(
            self.save_activations
        )

    def save_activations(self, module, inputs, output):
        # DenseNet의 이후 연산에서 원본 출력이 변경될 수 있으므로
        # clone한 텐서에 hook을 등록
        captured = output.clone()
        self.activations = captured.detach()
        captured.register_hook(self.save_gradients)
        return captured

    def save_gradients(self, grad):
        self.gradients = grad.detach()

    def generate(self, image, finding_idx, class_idx=1):
        self.model.eval()

        device = next(self.model.parameters()).device
        image = image.unsqueeze(0).to(device)

        self.model.zero_grad(set_to_none=True)
        self.activations = None
        self.gradients = None

        with torch.enable_grad():
            logits = self.model(image)

            score = logits[0, finding_idx, class_idx]
            score.backward()

        activations = self.activations
        gradients = self.gradients

        weights = gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * activations).sum(dim=1, keepdim=True)

        cam = F.relu(cam)
        cam = F.interpolate(
            cam,
            size=image.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        cam = cam[0, 0].detach().cpu().numpy()

        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)

        return cam

    def close(self):
        self.forward_handle.remove()

## 5. Grad-CAM 결과 시각화

하나의 X-ray 이미지에 대해 원본과 세 소견별 Grad-CAM Overlay를 나란히 출력한다.

- Original X-ray: 원본 흉부 X-ray
- Cardiomegaly: 심비대 양성 클래스 Grad-CAM Overlay
- Edema: 폐부종 양성 클래스 Grad-CAM Overlay
- Pleural Effusion: 흉수 양성 클래스 Grad-CAM Overlay

각 Overlay는 원본 이미지 위에 Grad-CAM 히트맵을 반투명하게 겹쳐 표시한다.

히트맵의 색상은 해당 클래스 점수에 대한 상대적 기여도를 나타낸다.

In [ ]:
def show_gradcam(image, cams):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    # 이미지 정규화 해제
    img = image.detach().cpu().permute(1, 2, 0).numpy()
    img = np.clip(img * std + mean, 0, 1)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    # 1. 원본 X-ray
    axes[0].imshow(img)
    axes[0].set_title("Original X-ray")

    # 2~4. 소견별 Grad-CAM Overlay
    for i, finding in enumerate(FINDING_NAMES):
        axes[i + 1].imshow(img)
        axes[i + 1].imshow(
            cams[i],
            cmap="jet",
            alpha=0.4,
            vmin=0,
            vmax=1
        )
        axes[i + 1].set_title(f"{finding} - Positive")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## 6. Grad-CAM 동작 테스트

학습된 DenseNet121 모델과 실제 test X-ray 이미지를 이용하여 Grad-CAM의 정상 동작을 확인한다.

### 확인 항목

- 세 소견에 대한 Grad-CAM 생성 여부
- 히트맵 출력 크기: (224, 224)
- 원본 이미지 및 소견별 Overlay 총 4개 출력 여부
- 소견별 Grad-CAM 생성 과정에서 오류 발생 여부

In [ ]:
from pathlib import Path

# 실행 환경에 맞게 경로를 수정하세요.
# PROJECT_DIR: 프로젝트 루트 폴더
# MODEL_PATH: 학습된 모델의 best_model.pth 파일 경로
# IMAGE_ROOT: X-ray JPG 이미지가 저장된 폴더
# TEST_CSV: 시각화에 사용할 test 이미지 목록 CSV
# 현재 MODEL_PATH와 IMAGE_ROOT는 GPU 서버 환경 기준입니다.
PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "gradcam":
    PROJECT_DIR = PROJECT_DIR.parent

# 성능평가에 사용한 모델과 동일하게 설정
MODEL_PATH = Path(
    "/home/team30103/chest-xray-multilabel/runs/pilot/weighted/"
    "train_20261015_112511_310261/best_model.pth"
)

IMAGE_ROOT = Path("/home/team30103/physionet_data")
TEST_CSV = (
    PROJECT_DIR / "data" / "processed" / "pilot" / "pilot_test.csv"
)

print("모델 파일 존재:", MODEL_PATH.is_file())
print("이미지 폴더 존재:", IMAGE_ROOT.is_dir())
print("test CSV 존재:", TEST_CSV.is_file())

In [ ]:
checkpoint = torch.load(
    MODEL_PATH,
    map_location="cpu",
    weights_only=True
)

# 소견·클래스 순서 확인
assert checkpoint["target_labels"] == FINDING_NAMES
assert checkpoint["class_names"] == [
    "negative", "positive", "uncertain"
]

if checkpoint.get("smoke_test", False):
    raise ValueError("임시 데이터로 학습한 demo 모델입니다.")

# 기존에 정의한 모델 구조에 학습된 가중치 적용
model = CXRModel(pretrained=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

print("사용 장치:", device)
print("저장 당시 epoch:", checkpoint["epoch"])
print("학습된 모델 불러오기 완료")

## 실제 X-ray 이미지 기반 Grad-CAM 동작 검증

GPU 서버에 저장된 MIMIC-CXR-JPG 이미지를 이용하여
Grad-CAM 시각화 코드의 정상 동작 여부를 확인한다.

학습된 모델을 사용하며, Grad-CAM은 해당 클래스 점수에 기여한 영역을 보여준다. 실제 병변 위치와 일치한다고 단정할 수는 없다.

데이터 접근 제한을 준수하기 위해 실제 X-ray 이미지 및
해당 이미지가 포함된 노트북 실행 결과는 GitHub에 업로드하지 않는다.

In [ ]:
import sys
import pandas as pd
from PIL import Image

# 학습 때 사용한 전처리 함수
sys.path.insert(0, str(PROJECT_DIR))
from cxr_preprocessing import preprocess_image

# test 이미지 목록
test_df = pd.read_csv(
    TEST_CSV,
    dtype={"dicom_id": "string"}
)

# 시각화할 이미지 선택: 0부터 시작
IMAGE_INDEX = 0
row = test_df.iloc[IMAGE_INDEX]

image_path = IMAGE_ROOT / f"{row['dicom_id']}.jpg"

if not image_path.is_file():
    raise FileNotFoundError(f"이미지가 없습니다: {image_path}")

# 학습과 동일한 흑백·패딩·리사이즈·정규화 적용
with Image.open(image_path) as original:
    image = preprocess_image(
        original,
        image_size=checkpoint["input"]["height"]
    )

print("선택한 이미지:", image_path)
print("이미지 형태:", image.shape)
print("소견 순서:", FINDING_NAMES)
print("원본 정답:", row[FINDING_NAMES].tolist())

In [ ]:
model.eval()
x = image.unsqueeze(0).to(device)

with torch.no_grad():
    # torchvision의 기존 forward
    original_logits = model.backbone(x).reshape(-1, 3, 3)

    # 수정한 forward
    updated_logits = model(x)

print(
    "최대 출력 차이:",
    (original_logits - updated_logits).abs().max().item()
)

print(
    "출력 일치:",
    torch.allclose(
        original_logits,
        updated_logits,
        atol=1e-6,
        rtol=1e-5
    )
)

In [ ]:
# 실제 X-ray 이미지 기반 Grad-CAM 테스트
gradcam = GradCAM(model)

try:
    cams = [
        gradcam.generate(image, finding_idx=i, class_idx=1)
        for i in range(3)
    ]

    show_gradcam(image, cams)

finally:
    gradcam.close()